In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("../data/processed/jogadores_saque.csv")

df.head()

In [ ]:
df_treino, df_teste = train_test_split(df, test_size=0.2, random_state=42)

print("Tamanho do Treino:", df_treino.shape)
print("Tamanho do Teste:", df_teste.shape)

In [ ]:
minha_f = 'pct_pontos_saque_ganhos ~ ht + age + surface + np.log(rank)'
modelo = smf.ols(formula=minha_f, data=df_treino)
modelo_treinado = modelo.fit()
modelo_treinado.summary()

In [ ]:
colunas_modelo = ['pct_pontos_saque_ganhos', 'ht', 'age', 'surface', 'rank']
df_teste_limpo = df_teste.dropna(subset=colunas_modelo)

previsoes = modelo_treinado.predict(df_teste_limpo)

erro_medio = mean_absolute_error(df_teste_limpo['pct_pontos_saque_ganhos'], previsoes)
r2_teste = r2_score(df_teste_limpo['pct_pontos_saque_ganhos'], previsoes)
residuos = df_teste_limpo['pct_pontos_saque_ganhos'] - previsoes

print(f"Erro médio absoluto: {erro_medio:.2f}")
print(f"R2 teste: {r2_teste:.2f}")
print(f"O quanto o modelo errou em cada partida de teste: {residuos}")

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1, 2, 1)
sns.histplot(residuos, kde=True, color='royalblue')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribuição dos Resíduos (Erros do modelo)')
plt.xlabel('Erro')
plt.ylabel('Frequência')

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1, 2, 2)
plt.scatter(previsoes, df_teste_limpo['pct_pontos_saque_ganhos'], alpha=0.1, color='purple')
plt.plot([40, 90], [40, 90], color='red', linestyle='--')
plt.title('Real vs Previsto')
plt.xlabel('Previsão do Model')
plt.ylabel('Valor Real')

In [ ]:
novos_jogadores = pd.DataFrame({
    'ht': [205, 178],
    'age': [26, 22],
    'surface': ['Grass', 'Grass'],
    'rank': [10, 80]
}, index=['Jogador A (Gigante - 2.05m)', 'Jogador B (Baixo - 1.71m)'])

novos_jogadores['Previsão_aproveitamento'] = modelo_treinado.predict(novos_jogadores)
novos_jogadores[['ht', 'surface', 'rank', 'Previsão_aproveitamento']]